In [66]:
from pathlib import Path
import duckdb

cwd = Path.cwd()
root = cwd.parent if cwd.name == 'notebooks' else cwd
db_path = root / "data" / "fdic.duckdb"
assert db_path.exists(), f"[duckdb_err]: path {db_path} not found"

with duckdb.connect(str(db_path), read_only=True) as conn:
    df = conn.sql("SELECT * from fct_bank_quarter_training").df()
    print(df.shape) 
    print(df.head())


(681297, 20)
    cert report_date bank_quarter_id  failure_in_year  \
0  57295  2014-12-31  57295_20141231                0   
1  57298  2014-12-31  57298_20141231                0   
2  57300  2014-12-31  57300_20141231                0   
3  57301  2014-12-31  57301_20141231                0   
4  57311  2014-12-31  57311_20141231                0   

   total_risk_based_capital_pct  nonperforming_assets_to_assets_pct  \
0                     14.639733                            0.000000   
1                     13.543432                            1.372407   
2                     13.349079                            1.833654   
3                     17.486149                            0.113230   
4                     13.555091                            0.106477   

   loan_loss_allowance_to_loans_pct   roa_pct  net_interest_margin_pct  \
0                          1.025175  1.719543                 1.922050   
1                          1.034782  0.736575                 4.66939

In [67]:
import pandas as pd

df['report_date'] = pd.to_datetime(df['report_date'])

train_split_dt = pd.Timestamp('2008-12-31')
test_split_dt = pd.Timestamp('2010-01-01')

train_df = df[df['report_date'] <= train_split_dt].copy()
test_df = df[df['report_date'] >= test_split_dt].copy()

print(f"Train dataframe: {train_df.shape}; Test dataframe: {test_df.shape}")

print(df['failure_in_year'].sum())
print(train_df['failure_in_year'].sum())
print(test_df['failure_in_year'].sum())


Train dataframe: (286499, 20); Test dataframe: (362408, 20)
2305
530
1109


In [68]:
feature_columns = [
    'nonperforming_assets_to_assets_pct', 
    'loan_loss_allowance_to_loans_pct', 
    'roa_pct', 
    'net_interest_margin_pct', 
    'efficiency_ratio_pct', 
    'brokered_deposits_pct', 
    'core_deposits_pct', 
    'fhlb_advances_to_assets_pct', 
    'construction_concentration_pct', 
    'leverage_ratio_pct', 
    'core_deposits_pct_4q_chg', 
    'nonperforming_assets_to_assets_pct_4q_chg', 
    'roa_pct_4q_chg', 
    'leverage_ratio_pct_4q_chg'
]

x_train = train_df[feature_columns]
y_train = train_df['failure_in_year']

x_test = test_df[feature_columns]
y_test = test_df['failure_in_year']


x_train_null_cnt = x_train.isnull().sum()
print(x_train_null_cnt)


nonperforming_assets_to_assets_pct               0
loan_loss_allowance_to_loans_pct                 0
roa_pct                                          0
net_interest_margin_pct                          0
efficiency_ratio_pct                             0
brokered_deposits_pct                          682
core_deposits_pct                                0
fhlb_advances_to_assets_pct                    601
construction_concentration_pct                 567
leverage_ratio_pct                               0
core_deposits_pct_4q_chg                     42516
nonperforming_assets_to_assets_pct_4q_chg    42516
roa_pct_4q_chg                               42516
leverage_ratio_pct_4q_chg                    42516
dtype: int64


In [69]:
x_train["is_first_year"] = (x_train["core_deposits_pct_4q_chg"].isna() & 
                            x_train["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_train["roa_pct_4q_chg"].isna() & 
                            x_train["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

x_test["is_first_year"] = (x_test["core_deposits_pct_4q_chg"].isna() & 
                            x_test["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_test["roa_pct_4q_chg"].isna() & 
                            x_test["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

In [70]:
x_train['nonperforming_assets_to_assets_pct'] = x_train['nonperforming_assets_to_assets_pct'].copy().fillna(x_train['nonperforming_assets_to_assets_pct'].median())
x_train['loan_loss_allowance_to_loans_pct'] = x_train['loan_loss_allowance_to_loans_pct'].copy().fillna(x_train['loan_loss_allowance_to_loans_pct'].median())
x_train['roa_pct'] = x_train['roa_pct'].copy().fillna(x_train['roa_pct'].median())
x_train['net_interest_margin_pct'] = x_train['net_interest_margin_pct'] .copy().fillna(x_train['net_interest_margin_pct'] .median())
x_train['efficiency_ratio_pct'] = x_train['efficiency_ratio_pct'].copy().fillna(x_train['efficiency_ratio_pct'].median())
x_train['brokered_deposits_pct'] = x_train['brokered_deposits_pct'].copy().fillna(x_train['brokered_deposits_pct'].median())
x_train['core_deposits_pct'] = x_train['core_deposits_pct'].copy().fillna(x_train['core_deposits_pct'].median())
x_train['fhlb_advances_to_assets_pct'] = x_train['fhlb_advances_to_assets_pct'].copy().fillna(x_train['fhlb_advances_to_assets_pct'].median())
x_train['construction_concentration_pct'] = x_train['construction_concentration_pct'].copy().fillna(x_train['construction_concentration_pct'].median())
x_train['leverage_ratio_pct'] = x_train['leverage_ratio_pct'].copy().fillna(x_train['leverage_ratio_pct'].median())
x_train['core_deposits_pct_4q_chg'] = x_train['core_deposits_pct_4q_chg'].copy().fillna(x_train['core_deposits_pct_4q_chg'].median())
x_train['nonperforming_assets_to_assets_pct_4q_chg'] = x_train['nonperforming_assets_to_assets_pct_4q_chg'].copy().fillna(x_train['nonperforming_assets_to_assets_pct_4q_chg'].median())
x_train['roa_pct_4q_chg'] = x_train['roa_pct_4q_chg'].copy().fillna(x_train['roa_pct_4q_chg'].median())
x_train['leverage_ratio_pct_4q_chg'] = x_train['leverage_ratio_pct_4q_chg'].copy().fillna(x_train['leverage_ratio_pct_4q_chg'].median())

In [71]:
x_test['nonperforming_assets_to_assets_pct']        = x_test['nonperforming_assets_to_assets_pct'].fillna(x_train['nonperforming_assets_to_assets_pct'].median())
x_test['loan_loss_allowance_to_loans_pct']          = x_test['loan_loss_allowance_to_loans_pct'].fillna(x_train['loan_loss_allowance_to_loans_pct'].median())
x_test['roa_pct']                                   = x_test['roa_pct'].fillna(x_train['roa_pct'].median())
x_test['net_interest_margin_pct']                   = x_test['net_interest_margin_pct'] .fillna(x_train['net_interest_margin_pct'] .median())
x_test['efficiency_ratio_pct']                      = x_test['efficiency_ratio_pct'].fillna(x_train['efficiency_ratio_pct'].median())
x_test['brokered_deposits_pct']                     = x_test['brokered_deposits_pct'].fillna(x_train['brokered_deposits_pct'].median())
x_test['core_deposits_pct']                         = x_test['core_deposits_pct'].fillna(x_train['core_deposits_pct'].median())
x_test['fhlb_advances_to_assets_pct']               = x_test['fhlb_advances_to_assets_pct'].fillna(x_train['fhlb_advances_to_assets_pct'].median())
x_test['construction_concentration_pct']            = x_test['construction_concentration_pct'].fillna(x_train['construction_concentration_pct'].median())
x_test['leverage_ratio_pct']                        = x_test['leverage_ratio_pct'].fillna(x_train['leverage_ratio_pct'].median())
x_test['core_deposits_pct_4q_chg']                  = x_test['core_deposits_pct_4q_chg'].fillna(x_train['core_deposits_pct_4q_chg'].median())
x_test['nonperforming_assets_to_assets_pct_4q_chg'] = x_test['nonperforming_assets_to_assets_pct_4q_chg'].fillna(x_train['nonperforming_assets_to_assets_pct_4q_chg'].median())
x_test['roa_pct_4q_chg']                            = x_test['roa_pct_4q_chg'].fillna(x_train['roa_pct_4q_chg'].median())
x_test['leverage_ratio_pct_4q_chg']                 = x_test['leverage_ratio_pct_4q_chg'].fillna(x_train['leverage_ratio_pct_4q_chg'].median())

In [72]:
print(f"x-train nulls: \n{x_train.isnull().sum()}\n")


x-train nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [73]:
print(f"x-test nulls: \n{x_test.isnull().sum()}\n")


x-test nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [74]:
print(f"medians: \n{x_train.median(numeric_only=True)}")

medians: 
nonperforming_assets_to_assets_pct            0.459944
loan_loss_allowance_to_loans_pct              1.212504
roa_pct                                       0.993106
net_interest_margin_pct                       4.008031
efficiency_ratio_pct                         65.853659
brokered_deposits_pct                         0.000000
core_deposits_pct                            69.014564
fhlb_advances_to_assets_pct                   1.545738
construction_concentration_pct                4.477114
leverage_ratio_pct                            9.397647
core_deposits_pct_4q_chg                     -0.552769
nonperforming_assets_to_assets_pct_4q_chg     0.000000
roa_pct_4q_chg                               -0.008681
leverage_ratio_pct_4q_chg                     0.029025
is_first_year                                 0.000000
dtype: float64
